In [1]:
import scipy.io
import os

print("--- INSPECTING .MAT FILE STRUCTURE ---")
sample_file = os.path.join("processed_files", "E1.mat")

try:
    # Load the MATLAB file
    mat_data = scipy.io.loadmat(sample_file)
    
    print(f"Keys found in {sample_file}:")
    # Print the variables ignoring the internal MATLAB metadata
    for key in mat_data:
        if not key.startswith('__'):
            data_type = type(mat_data[key])
            if hasattr(mat_data[key], 'shape'):
                print(f" -> '{key}': Shape {mat_data[key].shape} | Type: {mat_data[key].dtype}")
            else:
                print(f" -> '{key}': Type {data_type}")
except Exception as e:
    print(f"Failed to load .mat file. Error: {e}")
    print("We might need to use h5py if it's a v7.3 MAT file.")

--- INSPECTING .MAT FILE STRUCTURE ---
Keys found in processed_files\E1.mat:
 -> 'cores': Shape (1, 15791) | Type: object


In [2]:
import scipy.io
import os

mat_data = scipy.io.loadmat(os.path.join("processed_files", "E1.mat"))
cores = mat_data['cores'][0]  # shape (15791,)

print(f"Type of first element: {type(cores[0])}")
print(f"Shape of first element: {getattr(cores[0], 'shape', None)}")
print(f"First element preview:\n{cores[0]}")

Type of first element: <class 'numpy.ndarray'>
Shape of first element: (1, 4)
First element preview:
[[array([[(array([[array([[(array([[0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]]),)]],
                        dtype=[('data', 'O')])                                                ]],
                dtype=object),)                                                                  ]],
        dtype=[('nss', 'O')])
  array([[(array([[array([[(array([[0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]]),)]],
                        dtype=[('data', 'O')])                                                ]],
                dtype=object),)                                                                  ]],
        dtype=[('nss', 'O')])
  array([[(array([[array([[(array([[0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]]),)]],
                        dtype=[('data', 'O')])                                                ]],
                dtype=object),)                        

In [4]:
import numpy as np 
# Drill down to the CFR data for the first packet
first_packet = cores[0][0, 0]  # shape (1,4), take [0,0]
# The structure is: first_packet['nss'][0,0][0,0]['data'][0,0][0]
cfr_data = first_packet['nss'][0, 0][0, 0]['data'][0, 0][0]

print("CFR data shape:", cfr_data.shape)
print("CFR data dtype:", cfr_data.dtype)
print("First 10 subcarriers (complex):", cfr_data[:10])
print("Amplitude (abs) preview:", np.abs(cfr_data[:10]))

CFR data shape: (1024,)
CFR data dtype: complex128
First 10 subcarriers (complex): [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
Amplitude (abs) preview: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [5]:
import numpy as np
import pandas as pd
import scipy.io
import glob
import os
import warnings

warnings.filterwarnings('ignore')

def extract_ieee_dataset(data_dir, window_size=100, step_size=50):
    """
    Extracts rolling-window temporal features from the IEEE 802.11ax .mat files.
    - E files -> Label: 0 (Empty)
    - W, R, S files -> Label: 1 (Occupied)
    """
    mat_files = glob.glob(os.path.join(data_dir, '*.mat'))
    print(f"Found {len(mat_files)} .mat files. Starting extraction...\n")
    
    all_features = []
    all_labels = []
    
    for idx, filepath in enumerate(mat_files):
        filename = os.path.basename(filepath)
        
        # Determine Binary Occupancy Label
        if filename.startswith('E'): # Empty Space
            label = 0
        elif filename.startswith(('W', 'R', 'S')): # Human Present (Walking, Running, Staying)
            label = 1
        else:
            continue
            
        try:
            # 1. Load MAT file
            mat_data = scipy.io.loadmat(filepath)
            cores = mat_data['cores'][0] 
            
            # 2. Extract CFR Amplitudes for all packets
            packet_amplitudes = []
            
            # Sub-sample to speed up processing: extract every 10th packet
            for packet_idx in range(0, len(cores), 10): 
                try:
                    # Drill down into the MATLAB structure
                    cfr_complex = cores[packet_idx][0, 0]['nss'][0, 0][0, 0]['data'][0, 0][0]
                    # We take the absolute amplitude. Some edges are physically zeroed out in 802.11ax (1024 -> 996 usable).
                    packet_amplitudes.append(np.abs(cfr_complex))
                except IndexError:
                    pass # Skip corrupted packets
            
            # Stack into a 2D matrix: Shape (Num_Packets, 1024_subcarriers)
            amplitude_matrix = np.vstack(packet_amplitudes)
            num_packets = amplitude_matrix.shape[0]
            
            # Extract only the 996 usable subchannels (ignoring the dead control channels/null carriers)
            # The dead carriers are usually at the edges and exact center. 
            # To keep it simple but effective, we use all 1024 for the PCA to naturally discard the zeros.
            
            # 3. Slide a window to calculate Temporal Features
            file_features = []
            for start in range(0, num_packets - window_size, step_size):
                end = start + window_size
                window = amplitude_matrix[start:end, :] # Shape: (100, 1024)
                
                # Calculate the temporal variance of each subcarrier over this time window
                # High variance over time = Human moving/breathing. Zero variance = Empty room.
                temporal_variance = np.var(window, axis=0) 
                
                # We can also capture the mean amplitude of the window
                temporal_mean = np.mean(window, axis=0)
                
                # Combine them into a single 2048-length feature vector for this window slice
                combined_vector = np.concatenate((temporal_variance, temporal_mean))
                file_features.append(combined_vector)
            
            # 4. Append to master lists
            if file_features:
                X_file = np.vstack(file_features)
                y_file = np.full(X_file.shape[0], label)
                
                all_features.append(X_file)
                all_labels.append(y_file)
                
                print(f"[{idx+1}/{len(mat_files)}] {filename:<10} -> Label: {label} | Windows extracted: {X_file.shape[0]}")
                
        except Exception as e:
            print(f"Error on {filename}: {e}")
            continue

    # Final Stack
    X_final = np.vstack(all_features)
    y_final = np.concatenate(all_labels)
    
    return X_final, y_final

# Run the new Pipeline
data_folder = 'processed_files'
X, y = extract_ieee_dataset(data_folder, window_size=100, step_size=50)

print(f"\n✅ Total Dataset Shape: {X.shape[0]} samples, {X.shape[1]} features")
print(f"📊 Binary Class Balance (0=Empty, 1=Occupied):")
unique, counts = np.unique(y, return_counts=True)
print(dict(zip(unique, counts)))

Found 16 .mat files. Starting extraction...

[1/16] E1.mat     -> Label: 0 | Windows extracted: 30
[2/16] E2.mat     -> Label: 0 | Windows extracted: 30
[3/16] E3.mat     -> Label: 0 | Windows extracted: 30
[4/16] E4.mat     -> Label: 0 | Windows extracted: 30
[5/16] R1_P1.mat  -> Label: 1 | Windows extracted: 30
[6/16] R2_P1.mat  -> Label: 1 | Windows extracted: 29
[7/16] R3_P1.mat  -> Label: 1 | Windows extracted: 29
[8/16] R4_P1.mat  -> Label: 1 | Windows extracted: 30
[9/16] S1_P1.mat  -> Label: 1 | Windows extracted: 28
[10/16] S2_P1.mat  -> Label: 1 | Windows extracted: 29
[11/16] S3_P1.mat  -> Label: 1 | Windows extracted: 30
[12/16] S4_P1.mat  -> Label: 1 | Windows extracted: 29
[13/16] W1_P1.mat  -> Label: 1 | Windows extracted: 29
[14/16] W2_P1.mat  -> Label: 1 | Windows extracted: 30
[15/16] W3_P1.mat  -> Label: 1 | Windows extracted: 29
[16/16] W4_P1.mat  -> Label: 1 | Windows extracted: 28

✅ Total Dataset Shape: 470 samples, 2048 features
📊 Binary Class Balance (0=Empty, 

In [6]:
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

print("--- TRAINING IEEE 802.11ax PCA PIPELINE ---\n")

# 1. Stratified Split to maintain the 120 vs 350 balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train Set Shape: {X_train.shape}")
print(f"Test Set Shape:  {X_test.shape}\n")

# 2. Scale data (Mandatory for PCA & Distance-based algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Apply PCA to compress the massive 2048 temporal features
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA dramatically compressed 2048 temporal features into -> {X_train_pca.shape[1]} core components!\n")

# 4. Apply SMOTE because we have 3x more Occupied data than Empty data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_pca, y_train)

# ==========================================
# 🚀 TRAIN & EVALUATE MODELS
# ==========================================

models = {
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1)
}

results = []

for name, model in models.items():
    start = time.time()
    
    # Train
    model.fit(X_train_smote, y_train_smote)
    
    # Test
    preds = model.predict(X_test_pca)
    acc = accuracy_score(y_test, preds)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Train Time (s)": time.time() - start,
        "Raw Preds": preds # Store for confusion matrix
    })

# Format Output
print("="*60)
print("🏆 PART 1: BINARY OCCUPANCY DETECTION RESULTS 🏆")
print("="*60)
print(f"{'Model':<15} | {'Accuracy':<10} | {'Time (s)':<10}")
print("-" * 60)

best_model = None
best_acc = 0

for res in sorted(results, key=lambda x: x['Accuracy'], reverse=True):
    print(f"{res['Model']:<15} | {res['Accuracy']*100:>9.2f}% | {res['Train Time (s)']:>8.4f}s")
    if res['Accuracy'] > best_acc:
        best_acc = res['Accuracy']
        best_model = res

print("="*60)
print(f"\n--- DETAILED REPORT: {best_model['Model']} ---")
print(classification_report(y_test, best_model['Raw Preds'], target_names=['Empty (0)', 'Occupied (1)']))
print("\nConfusion Matrix:")
print("True\\Pred | Empty | Occupied")
cm = confusion_matrix(y_test, best_model['Raw Preds'])
print(f"Empty     | {cm[0][0]:<5} | {cm[0][1]}")
print(f"Occupied  | {cm[1][0]:<5} | {cm[1][1]}")

--- TRAINING IEEE 802.11ax PCA PIPELINE ---

Train Set Shape: (376, 2048)
Test Set Shape:  (94, 2048)

PCA dramatically compressed 2048 temporal features into -> 7 core components!

🏆 PART 1: BINARY OCCUPANCY DETECTION RESULTS 🏆
Model           | Accuracy   | Time (s)  
------------------------------------------------------------
Random Forest   |     98.94% |   0.2519s
LightGBM        |     98.94% |   2.6258s
XGBoost         |     97.87% |   0.0956s

--- DETAILED REPORT: Random Forest ---
              precision    recall  f1-score   support

   Empty (0)       1.00      0.96      0.98        24
Occupied (1)       0.99      1.00      0.99        70

    accuracy                           0.99        94
   macro avg       0.99      0.98      0.99        94
weighted avg       0.99      0.99      0.99        94


Confusion Matrix:
True\Pred | Empty | Occupied
Empty     | 23    | 1
Occupied  | 0     | 70
